In [0]:
%pip install xgboost shap --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score)
from xgboost import XGBClassifier
import mlflow

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
TABELA = f"{CATALOGO}.default.fase3_features_contexto"

In [0]:
FEATURES = [
    "qtd_redes",
    "percentual_participacao",
    "meta_do_ano",
    "taxa_2023",
    "media_portugues_2023",
    "participacao_2023",
    "gap_meta_2023",
    "esforco_necessario",
    "taxa_uf_2023",
    "posicao_relativa_uf",
    "meta_vs_uf",
    "tem_dado_completo",
    "flag_outlier",
]
TARGET = "target"
GRUPO = "id_municipio"

pdf = spark.table(TABELA).toPandas()

for c in ["tem_dado_completo", "flag_outlier"]:
    pdf[c] = pdf[c].astype(int)

X = pdf[FEATURES].copy()
y = pdf[TARGET].astype(int).copy()
groups = pdf[GRUPO].copy()

print(f"X: {X.shape} | features: {len(FEATURES)}")
print(f"Distribuição target:\n{y.value_counts(normalize=True).round(4)}")
print(f"Nulos:\n{X.isnull().sum()[X.isnull().sum() > 0]}")

In [0]:
cv = GroupKFold(n_splits=5)

modelos = {
    "baseline_dummy": Pipeline([
        ("clf", DummyClassifier(strategy="most_frequent", random_state=42))
    ]),
    "logistica": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, C=0.5, random_state=42,
                                   class_weight="balanced"))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=400, max_depth=6,
                                       min_samples_leaf=30, max_features="sqrt",
                                       random_state=42, class_weight="balanced",
                                       n_jobs=-1))
    ]),
    "xgboost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(n_estimators=400, max_depth=3,
                              learning_rate=0.03, subsample=0.8,
                              colsample_bytree=0.7, min_child_weight=10,
                              reg_lambda=2.0, reg_alpha=0.5,
                              random_state=42, eval_metric="logloss",
                              scale_pos_weight=(y == 0).sum() / (y == 1).sum()))
    ]),
}

metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]

In [0]:
mlflow.set_experiment("/Users/rm374983@fiap.com.br/fase3_alfabetizacao")

resultados = {}

for nome, modelo in modelos.items():
    with mlflow.start_run(run_name=f"{nome}_v2"):
        scores = cross_validate(modelo, X, y, groups=groups, cv=cv,
                                scoring=metricas, n_jobs=-1)
        res = {m: (scores[f"test_{m}"].mean(), scores[f"test_{m}"].std()) for m in metricas}
        resultados[nome] = res

        mlflow.log_params({"modelo": nome, "n_features": len(FEATURES), "cv": "GroupKFold-5"})
        for m, (mean, std) in res.items():
            mlflow.log_metric(f"{m}_mean", mean)
            mlflow.log_metric(f"{m}_std", std)

        print(f"\n=== {nome} ===")
        for m, (mean, std) in res.items():
            print(f"  {m:10s}: {mean:.4f} (+/- {std:.4f})")

In [0]:
comp = pd.DataFrame({
    nome: {m: f"{v[0]:.4f} ± {v[1]:.4f}" for m, v in res.items()}
    for nome, res in resultados.items()
}).T
display(comp)

In [0]:
melhor_nome = max(resultados, key=lambda k: resultados[k]["roc_auc"][0])
print(f"Melhor modelo por ROC-AUC: {melhor_nome}")

melhor = modelos[melhor_nome]
melhor.fit(X, y)

clf = melhor.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    imp = pd.DataFrame({"feature": FEATURES, "importancia": clf.feature_importances_})
else:
    imp = pd.DataFrame({"feature": FEATURES, "importancia": np.abs(clf.coef_[0])})
display(imp.sort_values("importancia", ascending=False))